# 🧠 Fine-Tuning Gemma 4 E2B for Empathetic Counseling

**Goal**: Fine-tune Google's Gemma 4 E2B (4-bit QLoRA) on MentalChat16K to build an emotionally intelligent counseling assistant.

**Stack**: Unsloth + TRL SFTTrainer + HuggingFace Datasets

**Hardware**: Google Colab T4 (free) or A100 (Pro)

**Portfolio Pitch**: _"Gemma 4 E2B fine-tuned for empathetic mental health support — bilingual Arabic/English inference ready"_

---

### What This Notebook Does
1. Installs Unsloth + dependencies
2. Loads Gemma 4 E2B in 4-bit quantization
3. Loads & preprocesses MentalChat16K (16K counselor-client Q&A pairs)
4. Converts to Gemma 4 chat template (system/user/assistant)
5. Trains with LoRA (rank 16)
6. Runs inference (English + Arabic)
7. Exports to GGUF + pushes to HuggingFace Hub

## 1. Install Dependencies

In [1]:
%%capture
!pip install "Pillow>=11.0.0" --force-reinstall
!pip install --upgrade --force-reinstall --no-cache-dir unsloth unsloth_zoo
!pip install --upgrade trl datasets huggingface_hub transformers accelerate bitsandbytes peft

In [2]:
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")

GPU: Tesla T4
VRAM: 15.6 GB
PyTorch: 2.10.0+cu128
CUDA: 12.8


## 2. Load Gemma 4 E2B (4-bit Quantized)

In [3]:
from unsloth import FastLanguageModel

max_seq_length = 2048  # Start conservative; E2B handles up to 32K

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Gemma-4-E2B-it",  # Unsloth's optimized variant
    max_seq_length=max_seq_length,
    load_in_4bit=True,                     # 4-bit quantization via bitsandbytes
    use_gradient_checkpointing="unsloth",  # Unsloth's custom GC — saves ~30% VRAM
)

print(f"Model loaded. Parameters: {model.num_parameters():,}")
print(f"Trainable (before LoRA): {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.2: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma4 won't work! Using float32.


Loading weights:   0%|          | 0/2011 [00:00<?, ?it/s]

Model loaded. Parameters: 5,123,178,016
Trainable (before LoRA): 0


## 3. Attach LoRA Adapters

In [4]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                          # LoRA rank — 16 is the sweet spot for E2B
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,                 # alpha = r → scaling factor = 1.0
    lora_dropout=0,                # Unsloth optimized — dropout=0 is faster
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    max_seq_length=max_seq_length,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = model.num_parameters()
print(f"Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

Unsloth: Making `model.base_model.model.model.language_model` require gradients
Trainable params: 31,039,488 / 5,154,217,504 (0.60%)


## 4. Load & Preprocess MentalChat16K

MentalChat16K has two CSV files:
- `Synthetic_Data_10K.csv` — 9,775 GPT-generated counselor–client conversations
- `Interview_Data_6K.csv` — 6,338 real clinical trial Q&A pairs

Both have `input` (client question) and `output` (counselor response) columns.

In [5]:
from datasets import load_dataset, concatenate_datasets
import pandas as pd

# Load both splits from the HF repo
# MentalChat16K stores data as CSV files in the repo
synthetic = load_dataset(
    "ShenLab/MentalChat16K",
    data_files="Synthetic_Data_10K.csv",
    split="train"
)
interview = load_dataset(
    "ShenLab/MentalChat16K",
    data_files="Interview_Data_6K.csv",
    split="train"
)

print(f"Synthetic: {len(synthetic)} rows | Columns: {synthetic.column_names}")
print(f"Interview: {len(interview)} rows | Columns: {interview.column_names}")

# Preview
print("\n--- Sample (Synthetic) ---")
print(f"Input: {synthetic[0][synthetic.column_names[0]][:200]}...")
print(f"Output: {synthetic[0][synthetic.column_names[1]][:200]}...")

Synthetic: 9774 rows | Columns: ['instruction', 'input', 'output']
Interview: 6310 rows | Columns: ['instruction', 'input', 'output']

--- Sample (Synthetic) ---
Input: You are a helpful mental health counselling assistant, please answer the mental health questions based on the patient's description. 
The assistant gives helpful, comprehensive, and appropriate answer...
Output: I think I might be developing a substance abuse problem. Lately, I've been relying on alcohol as a way to cope with my emotions and escape from reality. It started off as occasional drinking, but now ...


In [6]:
print("=== COLUMNS ===")
for col in synthetic.column_names:
    print(f"\n--- {col} ---")
    print(str(synthetic[0][col])[:300])

=== COLUMNS ===

--- instruction ---
You are a helpful mental health counselling assistant, please answer the mental health questions based on the patient's description. 
The assistant gives helpful, comprehensive, and appropriate answers to the user's questions. 

--- input ---
I think I might be developing a substance abuse problem. Lately, I've been relying on alcohol as a way to cope with my emotions and escape from reality. It started off as occasional drinking, but now it has become a daily habit. I'm worried about the impact it's having on my physical and mental well

--- output ---
I'm really glad that you reached out and shared what you've been going through. It takes a lot of courage to acknowledge and address these concerns. I can understand how relying on alcohol as a coping mechanism might feel like an escape from reality, but it's important to remember that there are hea


In [7]:
# Common column names in MentalChat16K (adjust after inspection)
INPUT_COL = "input"    # client message  — may be "question" or "input"
OUTPUT_COL = "output"  # counselor reply  — may be "answer" or "output"

# Normalize column names across both splits
def normalize_columns(ds, input_col, output_col):
    """Rename to standard 'input'/'output' if needed."""
    if input_col != "input" and input_col in ds.column_names:
        ds = ds.rename_column(input_col, "input")
    if output_col != "output" and output_col in ds.column_names:
        ds = ds.rename_column(output_col, "output")
    return ds

synthetic = normalize_columns(synthetic, INPUT_COL, OUTPUT_COL)
interview = normalize_columns(interview, INPUT_COL, OUTPUT_COL)

# Combine into one dataset
# Keep only the columns we need
cols_to_keep = ["input", "output"]
synthetic = synthetic.select_columns([c for c in cols_to_keep if c in synthetic.column_names])
interview = interview.select_columns([c for c in cols_to_keep if c in interview.column_names])

dataset = concatenate_datasets([synthetic, interview])
print(f"\nCombined dataset: {len(dataset)} rows")


Combined dataset: 16084 rows


In [8]:
# Filter out empty or very short examples
dataset = dataset.filter(
    lambda x: (
        x["input"] is not None and
        x["output"] is not None and
        len(str(x["input"]).strip()) > 20 and
        len(str(x["output"]).strip()) > 20
    )
)
print(f"After filtering: {len(dataset)} rows")

After filtering: 16052 rows


## 5. Format for Gemma 4 Chat Template

Gemma 4 uses standard `system` / `user` / `assistant` roles.

Key rules from the Unsloth guide:
- Use standard chat roles (not older Gemma-specific formats)
- For thinking mode: prefix system prompt with `<|think|>` (we'll skip this)
- Multi-turn: only keep the final visible answer

In [9]:
SYSTEM_PROMPT = """You are a compassionate and emotionally intelligent counseling assistant. \
You provide empathetic, supportive, and non-judgmental responses to people \
seeking help with mental health challenges including anxiety, depression, \
relationships, grief, and emotional wellbeing. You listen actively, validate \
feelings, and offer constructive coping strategies. You respond in the same \
language the user writes in."""


def format_to_chat(example):
    """Convert Q&A pair to Gemma 4 chat format."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": str(example["input"]).strip()},
        {"role": "assistant", "content": str(example["output"]).strip()},
    ]
    # Apply the tokenizer's chat template
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}


dataset = dataset.map(format_to_chat, num_proc=4)

# Preview a formatted example
print("=" * 60)
print("FORMATTED EXAMPLE:")
print("=" * 60)
print(dataset[0]["text"][:1000])
print("...")

FORMATTED EXAMPLE:
<bos><|turn>system
You are a compassionate and emotionally intelligent counseling assistant. You provide empathetic, supportive, and non-judgmental responses to people seeking help with mental health challenges including anxiety, depression, relationships, grief, and emotional wellbeing. You listen actively, validate feelings, and offer constructive coping strategies. You respond in the same language the user writes in.<turn|>
<|turn>user
I think I might be developing a substance abuse problem. Lately, I've been relying on alcohol as a way to cope with my emotions and escape from reality. It started off as occasional drinking, but now it has become a daily habit. I'm worried about the impact it's having on my physical and mental well-being, as well as my relationships and work performance.<turn|>
<|turn>model
I'm really glad that you reached out and shared what you've been going through. It takes a lot of courage to acknowledge and address these concerns. I can under

In [10]:
import numpy as np

# Gemma 4 returns a Processor; access the underlying tokenizer
tok = tokenizer.tokenizer if hasattr(tokenizer, 'tokenizer') else tokenizer

lengths = [len(tok.encode(x["text"])) for x in dataset.select(range(min(500, len(dataset))))]
print(f"Token length stats (sample of {len(lengths)}):")
print(f"  Mean: {np.mean(lengths):.0f}")
print(f"  Median: {np.median(lengths):.0f}")
print(f"  95th pctl: {np.percentile(lengths, 95):.0f}")
print(f"  Max: {max(lengths)}")
print(f"\nmax_seq_length is set to {max_seq_length} — adjust if 95th pctl exceeds this.")

Token length stats (sample of 500):
  Mean: 537
  Median: 536
  95th pctl: 629
  Max: 682

max_seq_length is set to 2048 — adjust if 95th pctl exceeds this.


## 6. Train with SFTTrainer

In [11]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        # === Core training ===
        max_seq_length=max_seq_length,
        per_device_train_batch_size=1,       # Drop to 1
        gradient_accumulation_steps=8,       # Keep effective batch = 8

        # === Schedule ===
        max_steps=200,                       # ~1,600 examples — enough for portfolio
        warmup_steps=20,
        learning_rate=5e-5,
        lr_scheduler_type="cosine",

        # === Precision & optimization ===
        fp16=False,
        bf16=False,                          # Let Unsloth handle precision
        optim="adamw_8bit",
        weight_decay=0.01,
        max_grad_norm=0.3,                   # Tighter clipping

        # === Logging ===
        logging_steps=10,
        save_steps=100,
        save_total_limit=2,
        output_dir="./outputs_gemma4_eq",

        # === Dataset ===
        dataset_text_field="text",
        dataset_num_proc=2,
        packing=False,

        seed=3407,
    ),
)

print(f"Training samples: {len(dataset)}")
print(f"Max steps: 200")
print(f"Effective examples: ~1,600")

Unsloth: Switching to float32 training since model cannot work with float16


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/16052 [00:00<?, ? examples/s]

Training samples: 16052
Max steps: 200
Effective examples: ~1,600


In [12]:
# ============================
#         TRAIN!
# ============================
gpu_stats_before = torch.cuda.get_device_properties(0)
start_mem = torch.cuda.memory_allocated() / 1e9

trainer_stats = trainer.train()

end_mem = torch.cuda.memory_allocated() / 1e9
print(f"\n{'='*40}")
print(f"Training complete!")
print(f"Training loss: {trainer_stats.training_loss:.4f}")
print(f"Runtime: {trainer_stats.metrics['train_runtime']/60:.1f} min")
print(f"VRAM used: {end_mem:.1f} GB")
print(f"{'='*40}")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 16,052 | Num Epochs = 1 | Total steps = 200
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 31,039,488 of 5,154,217,504 (0.60% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
10,12.683388
20,8.936349
30,6.520969
40,5.883101
50,7.568495
60,11.068379
70,12.122606
80,12.276423
90,12.321781
100,12.340549



Training complete!
Training loss: 11.2656
Runtime: 32.1 min
VRAM used: 7.6 GB


## 7. Inference — Test the Fine-Tuned Model

Test in both English and Arabic to validate bilingual capability.

In [15]:
FastLanguageModel.for_inference(model)

def chat(user_message, max_new_tokens=512):
    """Send a message and get the model's response."""
    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {"role": "user", "content": [{"type": "text", "text": user_message}]},
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(model.device)

    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=max_new_tokens,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.1,
        do_sample=True,
    )
    response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
    return response.strip()

In [16]:
# === English Test Cases ===
test_prompts_en = [
    "I've been feeling really anxious about starting a new job. I can't sleep and I keep thinking about everything that could go wrong.",
    "My best friend and I had a huge fight and I don't know if our friendship can recover. I feel so alone.",
    "I lost my father last month and I don't know how to cope. Some days I feel numb, other days I can't stop crying.",
]

print("=" * 60)
print("ENGLISH INFERENCE")
print("=" * 60)
for i, prompt in enumerate(test_prompts_en, 1):
    print(f"\n--- Test {i} ---")
    print(f"User: {prompt}")
    print(f"\nAssistant: {chat(prompt)}")
    print()

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


ENGLISH INFERENCE

--- Test 1 ---
User: I've been feeling really anxious about starting a new job. I can't sleep and I keep thinking about everything that could go wrong.

Assistant: It sounds like you're carrying a lot of worry right now, and it's completely understandable that you're feeling anxious about starting a new job. Those kinds of feelings—the racing thoughts and the trouble sleeping—can be really overwhelming. Please know that what you're experiencing is a very common reaction to a big life change, and it's okay to feel this way.

I'm here to listen without judgment. If you'd like, we can talk more about what's going through your mind.

To start, would you like to tell me a little more about what specifically is making you anxious? For example:

* **What are some of the specific things you are worried about happening at the new job?** (e.g., performance, fitting in, social situations)
* **How long have you been feeling this way?**
* **Are there any particular thoughts or sc

In [17]:
# === Arabic Test Cases ===
# Gemma 4 natively supports Arabic — let's test cross-lingual transfer
test_prompts_ar = [
    "أشعر بالقلق الشديد من المستقبل ولا أستطيع النوم. ماذا أفعل؟",
    "أنا حزين جداً بعد فقدان والدي. كيف أتعامل مع هذا الألم؟",
    "أشعر بالوحدة وليس لدي أصدقاء أتحدث إليهم. هل من نصيحة؟",
]

print("=" * 60)
print("ARABIC INFERENCE (Cross-lingual transfer)")
print("=" * 60)
for i, prompt in enumerate(test_prompts_ar, 1):
    print(f"\n--- Test {i} ---")
    print(f"User: {prompt}")
    print(f"\nAssistant: {chat(prompt)}")
    print()

ARABIC INFERENCE (Cross-lingual transfer)

--- Test 1 ---
User: أشعر بالقلق الشديد من المستقبل ولا أستطيع النوم. ماذا أفعل؟

Assistant: أتفهم تمامًا مدى صعوبة هذا الشعور. القلق بشأن المستقبل وصعوبة النوم يمكن أن يكونا مرهقين جدًا ومؤلمين في آن واحد. أريدك أن تعرفي أن مشاعرك هذه طبيعية جدًا، والكثير من الناس يمرون بهذه التجربة. أنا هنا لأستمع إليك وأدعمك بكل ما أستطيع.

**دعيني أقول لكِ أولاً:** خذي نفسًا عميقًا وبطيء. حاولي أن تأخذي بضع لحظات الآن للتركيز على تنفسك. لا تضغطي على نفسك لتشعري بتحسن فوري.

**إليكِ بعض الأشياء التي قد تساعدك الآن، ويمكنك تجربتها:**

1. **الاعتراف بالمشاعر (Validation):**
   * **قولي لنفسك:** "أنا أشعر بالقلق الشديد بشأن المستقبل، وهذا طبيعي. من الطبيعي أن أكون قلقة عندما أفكر في المجهول."
   * **التعاطف مع الذات:** عاملِي نفسك بلطف كما لو كنتِ تعاملين صديقة مقربة تمر بوقت صعب. لا تلومي نفسك على شعورك بالقلق.

2. **تقنيات التأريض (Grounding Techniques) للمساعدة على تهدئة العقل:**
   * **تقنية 5-4-3-2-1:** هذه التقنية مفيدة لإعادة عقلك إلى اللحظة الحالية بدل

## 8. Save & Export

### 8a. Save LoRA Adapters (lightweight — just the adapters)

In [18]:
# Save locally
model.save_pretrained("gemma4-e2b-eq-lora")
tokenizer.save_pretrained("gemma4-e2b-eq-lora")
print("LoRA adapters saved to ./gemma4-e2b-eq-lora")

LoRA adapters saved to ./gemma4-e2b-eq-lora


### 8b. Push to HuggingFace Hub

In [19]:
# === CONFIGURE THESE ===
HF_USERNAME = "muaaz"   # <-- Change this
MODEL_NAME = "gemma4-e2b-eq-counselor"
REPO_ID = f"{HF_USERNAME}/{MODEL_NAME}"

# Login (will prompt for token)
from huggingface_hub import login
login()  # Paste your HF token when prompted

In [21]:
# Push LoRA adapters
model.push_to_hub(REPO_ID)
tokenizer.push_to_hub(REPO_ID)
print(f"\nPushed to: https://huggingface.co/{REPO_ID}")

README.md:   0%|          | 0.00/533 [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          |  597kB /  124MB            

Saved model to https://huggingface.co/muaaz/gemma4-e2b-eq-counselor


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mp16yqbwvi/tokenizer.json:   1%|          |  218kB / 32.2MB            


Pushed to: https://huggingface.co/muaaz/gemma4-e2b-eq-counselor


### 8c. Export to GGUF (for Ollama / llama.cpp deployment)

In [ ]:
# Save as GGUF — choose your quantization level
# q4_k_m: Best balance of size vs quality for deployment
# q8_0:   Higher quality, larger file
# f16:    Full precision, largest

model.save_pretrained_gguf(
    "gemma4-e2b-eq-gguf",
    tokenizer,
    quantization_method="q4_k_m",
)
print("GGUF saved to ./gemma4-e2b-eq-gguf")

# Optionally push GGUF to HuggingFace too
model.push_to_hub_gguf(REPO_ID + "-GGUF", tokenizer, quantization_method="q4_k_m")

## 9. Model Card (for HuggingFace Hub)

Copy the markdown below into your HuggingFace model's README.md:

In [ ]:
model_card = """
---
language:
- en
- ar
license: apache-2.0
tags:
- gemma4
- mental-health
- counseling
- emotional-intelligence
- arabic
- bilingual
- lora
- unsloth
datasets:
- ShenLab/MentalChat16K
base_model: google/gemma-4-e2b-it
---

# Gemma 4 E2B — Empathetic Counseling Assistant

Fine-tuned **Gemma 4 E2B** for empathetic mental health counseling support.
Trained on **MentalChat16K** (16K counselor–client conversations covering 33 mental health topics).

## Key Features
- **Empathetic responses** across anxiety, depression, grief, relationships, and more
- **Bilingual** — responds in English and Arabic (leveraging Gemma 4's native 140-language support)
- **Lightweight** — LoRA adapters (only ~0.5% of parameters trained)
- **Deployable** — GGUF export available for Ollama / llama.cpp

## Training Details
- **Base model**: google/gemma-4-e2b-it
- **Method**: QLoRA (4-bit, rank 16, alpha 16)
- **Dataset**: ShenLab/MentalChat16K (16,113 Q&A pairs)
- **Framework**: Unsloth + TRL SFTTrainer
- **Hardware**: Google Colab (T4 16GB / A100 40GB)

## Disclaimer
This model is for research and educational purposes only.
It is NOT a substitute for professional mental health care.
If you or someone you know is in crisis, please contact a licensed professional.
"""

with open("gemma4-e2b-eq-lora/README.md", "w") as f:
    f.write(model_card)

print("Model card written to gemma4-e2b-eq-lora/README.md")

## 10. Quick Troubleshooting

| Issue | Fix |
|-------|-----|
| OOM on T4 | Set `per_device_train_batch_size=1`, reduce `max_seq_length` to 1024 |
| Loss doesn't decrease | Check data formatting — print `dataset[0]['text']` and verify chat template |
| Arabic output is garbage | E2B should handle Arabic natively; verify the system prompt isn't forcing English |
| GGUF export fails | Update unsloth: `pip install --upgrade unsloth unsloth_zoo` |
| Column name errors | Run the inspection cell (Section 4) and update `INPUT_COL` / `OUTPUT_COL` |
| Model generates endlessly | Ensure EOS token is correctly set — Unsloth handles this automatically |